<div>
<img src="img/NumPy_logo.png" width="500"/>
</div>

# Introduction à NumPy — Analyse de données météo

Quelle que soit la nature des données, elles peuvent souvent être représentées sous forme de tableaux. Par exemple, une image est un tableau à 2 dimensions contenant l'intensité  d'un pixel. Il est fondamental de pouvoir manipuler efficacement ces tableaux. 

En python, on utilise pour cela [Numpy (Numerical Python)](https://numpy.org/)  (*Numerical Python*) qui fournit une interface pour stocker et effectuer des opérations sur les données de manière efficace, même sur les tableaux de large taille. 

Le cœur de la philosophie de Numpy réside dans l’utilisation d’<span style="color:blue">_objets de type tableau (array) multidimensionnel_</span>, que l’on qualifiera de <span style="color:blue">_vecteurs_</span> (ligne ou colonne de taille n, une dimension) ou de <span style="color:blue">_matrices_</span> (tableau à 2 dimensions de tailles n×m, multidimensionnel). 

Il existe de nombreuses <span style="color:blue">_fonctions pré-compilées et optimisées_</span> associées à ce type qui permettent de simplifier l’écriture du code et en accélérer l'exécution. 

Dans ce TP, nous allons manipuler des **données météo** : températures, précipitations et mesures effectuées
dans plusieurs villes et sur plusieurs jours.


## Objectifs

À la fin de ce TP, vous saurez :

- créer des tableaux NumPy et comprendre leur `dtype` ;
- manipuler des tableaux à une ou plusieurs dimensions ;
- accéder à des éléments ou à des sous-tableaux ;
- concaténer des données ;
- effectuer des calculs vectorisés sans boucle Python ;
- filtrer des valeurs avec des masques et `np.where` ;
- calculer des statistiques avec `axis` ;
- utiliser le **broadcasting**.

In [62]:
# Importation de NumPy
import numpy as np

# Graine fixée pour obtenir les mêmes résultats à chaque exécution
rng = np.random.default_rng(42)

## 2. Type des tableaux

Un tableau NumPy contient des éléments d'un **même type**.
Le type (`dtype`) est généralement déduit lors de la création du tableau, mais il peut aussi être imposé.
Le type a un impact sur les opérations qui sont appliquées au tableau.

Nous allons commencer avec quelques relevés de température.

In [2]:
# Températures maximales relevées sur 5 jours (en °C)
temperatures = np.array([18, 21, 19, 24, 22])
print(temperatures)

[18 21 19 24 22]


<span style="color:blue">**QUESTION : Quel est le type (`dtype`) du tableau `temperatures` ?**</span>

<span style="color:red">**CORRECTION :** Le tableau ne contient que des entiers : NumPy choisit donc un type entier (généralement `int64` ou `int32` selon la plateforme).</span>

In [3]:
print(temperatures.dtype)

int64


In [4]:
# Une mesure comporte maintenant une décimale
temperatures_precises = np.array([18.5, 21, 19, 24, 22])
print(temperatures_precises)

[18.5 21.  19.  24.  22. ]


<span style="color:blue">**QUESTION : Quel est le type de `temperatures_precises` ? Pourquoi les valeurs entières apparaissent-elles comme des nombres décimaux ?**</span>

<span style="color:red">**CORRECTION :** Le tableau est de type flottant : la présence de `18.5` conduit NumPy à convertir les autres valeurs en nombres à virgule flottante.</span>

In [5]:
print(temperatures_precises.dtype)

float64


In [6]:
# On peut imposer explicitement le type
precipitations = np.array([0, 2, 5, 1], dtype="float32")
print(precipitations)

[0. 2. 5. 1.]


<span style="color:blue">**QUESTION : Quel est le type de `precipitations` ?**</span>

<span style="color:red">**CORRECTION :** Il s'agit de `float32`, car ce type a été imposé avec l'argument `dtype`.</span>

In [7]:
print(precipitations.dtype)

float32


## 3. Tableaux multidimensionnels

Les données météo sont naturellement multidimensionnelles.
On peut par exemple organiser un tableau avec :

- une **ligne par ville** ;
- une **colonne par jour**.

Voici trois séries de températures construites avec `range`.

In [8]:
exemple = np.array([range(i, i + 4) for i in [15, 18, 21]])
print(exemple)

[[15 16 17 18]
 [18 19 20 21]
 [21 22 23 24]]


Les cellules suivantes présentent plusieurs fonctions utiles pour créer rapidement des tableaux.
Vous pouvez modifier leurs paramètres pour observer leur comportement.

In [9]:
# 7 jours sans précipitations
print(np.zeros(7, dtype=int))

[0 0 0 0 0 0 0]


In [10]:
# Prévision identique de 20 °C pour 3 villes pendant 5 jours
print(np.full((3, 5), 20.0))

[[20. 20. 20. 20. 20.]
 [20. 20. 20. 20. 20.]
 [20. 20. 20. 20. 20.]]


In [11]:
# Tableau de coefficients initialisés à 1
print(np.ones((3, 5), dtype=float))

[[1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]]


In [12]:
# Heures de relevé : de 0 h à 21 h, toutes les 3 heures
print(np.arange(0, 24, 3))

[ 0  3  6  9 12 15 18 21]


In [13]:
# 5 valeurs régulièrement espacées entre 10 °C et 30 °C
print(np.linspace(10, 30, 5))

[10. 15. 20. 25. 30.]


In [14]:
# Simulation de 3 x 3 mesures aléatoires entre 0 et 1
print(rng.random((3, 3)))

[[0.77395605 0.43887844 0.85859792]
 [0.69736803 0.09417735 0.97562235]
 [0.7611397  0.78606431 0.12811363]]


<span style="color:blue">**QUESTION : Créer une matrice identité de taille 4 × 4 avec `np.eye`.**</span>

<span style="color:red">**CORRECTION :** `np.eye(4)` crée une matrice carrée 4 × 4 contenant des 1 sur la diagonale et des 0 ailleurs.</span>

In [15]:
print(np.eye(4))

[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


<span style="color:blue">**QUESTION : Créer, avec `np.arange` puis `reshape`, un tableau 4 × 4 contenant les entiers de 0 à 15.**</span>

<span style="color:red">**CORRECTION :** `np.arange(16).reshape(4, 4)` produit 16 valeurs puis les réorganise en 4 lignes et 4 colonnes.</span>

In [16]:
grille = np.arange(16).reshape(4, 4)
print(grille)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]]


## 4. Obtenir les caractéristiques d'un tableau

Les propriétés importantes d'un tableau NumPy sont notamment :

- `ndim` : nombre de dimensions ;
- `shape` : forme du tableau ;
- `size` : nombre total d'éléments ;
- `dtype` : type des données.

In [17]:
# Températures d'une ville sur 7 jours
semaine = np.array([18, 20, 21, 19, 23, 25, 24])

print(semaine)
print("Nombre de dimensions :", semaine.ndim)
print("Forme :", semaine.shape)
print("Nombre d'éléments :", semaine.size)
print("Type :", semaine.dtype)

[18 20 21 19 23 25 24]
Nombre de dimensions : 1
Forme : (7,)
Nombre d'éléments : 7
Type : int64


In [18]:
# Températures de 3 villes sur 5 jours
temperatures_villes = np.array([
    [18.5, 20.0, 21.5, 23.0, 22.0],  # Lille
    [22.0, 24.5, 26.0, 27.5, 25.0],  # Lyon
    [25.0, 27.0, 29.5, 31.0, 30.0]   # Montpellier
])

print(temperatures_villes)
print("Nombre de dimensions :", temperatures_villes.ndim)
print("Forme :", temperatures_villes.shape)
print("Nombre d'éléments :", temperatures_villes.size)
print("Type :", temperatures_villes.dtype)

[[18.5 20.  21.5 23.  22. ]
 [22.  24.5 26.  27.5 25. ]
 [25.  27.  29.5 31.  30. ]]
Nombre de dimensions : 2
Forme : (3, 5)
Nombre d'éléments : 15
Type : float64


<span style="color:blue">**QUESTION : Que signifie la forme `(3, 5)` de `temperatures_villes` dans le contexte de nos données ?**</span>

<span style="color:red">**CORRECTION :** Le tableau contient 3 lignes (les 3 villes) et 5 colonnes (les 5 jours de mesure).</span>

## 5. Accéder à un élément d'un tableau

On accède aux éléments avec les crochets `[]`.

Pour un tableau à une dimension :

- `T[0]` : premier élément ;
- `T[-1]` : dernier élément ;
- `T[2]` : élément d'indice 2, donc troisième élément.

Pour un tableau à deux dimensions, on précise généralement **ligne puis colonne** :
`T[ligne, colonne]`.

In [19]:
temperatures_paris = np.array([17, 19, 21, 20, 23, 25, 22])
print(temperatures_paris)

[17 19 21 20 23 25 22]


<span style="color:blue">**QUESTION : Accéder à la température du premier jour.**</span>

<span style="color:red">**CORRECTION :** `temperatures_paris[0]` sélectionne le premier élément.</span>

In [20]:
print(temperatures_paris[0])

17


<span style="color:blue">**QUESTION : Accéder à la température du dernier jour.**</span>

<span style="color:red">**CORRECTION :** `temperatures_paris[-1]` sélectionne le dernier élément.</span>

In [21]:
print(temperatures_paris[-1])

22


<span style="color:blue">**QUESTION : Quelle est la longueur de `temperatures_paris` ?**</span>

<span style="color:red">**CORRECTION :** La longueur vaut 7 : il y a une température par jour sur une semaine.</span>

In [22]:
print(len(temperatures_paris))

7


In [23]:
# Lignes : Lille, Lyon, Montpellier
# Colonnes : lundi, mardi, mercredi, jeudi, vendredi
temperatures_villes = np.array([
    [18.5, 20.0, 21.5, 23.0, 22.0],
    [22.0, 24.5, 26.0, 27.5, 25.0],
    [25.0, 27.0, 29.5, 31.0, 30.0]
])
print(temperatures_villes)

[[18.5 20.  21.5 23.  22. ]
 [22.  24.5 26.  27.5 25. ]
 [25.  27.  29.5 31.  30. ]]


<span style="color:blue">**QUESTION : Accéder à la température de Montpellier le mercredi.**</span>

<span style="color:red">**CORRECTION :** Montpellier est sur la ligne d'indice 2 et mercredi sur la colonne d'indice 2 : `temperatures_villes[2, 2]`.</span>

In [24]:
print(temperatures_villes[2, 2])

29.5


## 6. Accéder à plusieurs éléments : slicing

La syntaxe générale est :

`x[début:fin:pas]`

La borne `fin` n'est **pas incluse**.
Le début, la fin ou le pas peuvent être omis.

In [25]:
print(temperatures_paris)

[17 19 21 20 23 25 22]


<span style="color:blue">**QUESTION : Afficher les 5 premiers jours.**</span>

<span style="color:red">**CORRECTION :** `temperatures_paris[:5]` sélectionne les indices 0 à 4.</span>

In [26]:
print(temperatures_paris[:5])

[17 19 21 20 23]


<span style="color:blue">**QUESTION : Afficher les températures à partir du 5ᵉ jour (indice 4 inclus).**</span>

<span style="color:red">**CORRECTION :** `temperatures_paris[4:]` sélectionne les éléments à partir de l'indice 4.</span>

In [27]:
print(temperatures_paris[4:])

[23 25 22]


<span style="color:blue">**QUESTION : Afficher une température sur deux.**</span>

<span style="color:red">**CORRECTION :** `temperatures_paris[::2]` utilise un pas de 2.</span>

In [28]:
print(temperatures_paris[::2])

[17 21 23 22]


Un pas négatif permet de parcourir un tableau à l'envers.

In [29]:
print("Ordre normal :", temperatures_paris)
print("Ordre inversé :", temperatures_paris[::-1])

Ordre normal : [17 19 21 20 23 25 22]
Ordre inversé : [22 25 23 20 21 19 17]


Avec un tableau à deux dimensions, on peut extraire une ligne ou une colonne complète.

In [30]:
print(temperatures_villes)
print("Première ville :", temperatures_villes[0, :])

[[18.5 20.  21.5 23.  22. ]
 [22.  24.5 26.  27.5 25. ]
 [25.  27.  29.5 31.  30. ]]
Première ville : [18.5 20.  21.5 23.  22. ]


<span style="color:blue">**QUESTION : Afficher toutes les températures du mercredi.**</span>

<span style="color:red">**CORRECTION :** Il faut récupérer toutes les lignes (`:`) de la colonne d'indice 2 : `temperatures_villes[:, 2]`.</span>

In [31]:
print(temperatures_villes[:, 2])

[21.5 26.  29.5]


## 7. Concaténer des tableaux

NumPy permet de réunir plusieurs tableaux avec `np.concatenate`, `np.vstack`, `np.hstack`, etc.

In [32]:
# Deux demi-semaines pour une même ville
debut_semaine = np.array([18, 20, 21])
fin_semaine = np.array([22, 24, 23, 19])

semaine_complete = np.concatenate([debut_semaine, fin_semaine])
print(semaine_complete)

[18 20 21 22 24 23 19]


In [33]:
# Ajout d'une nouvelle ville au tableau existant
nouvelle_ville = np.array([19.0, 21.0, 22.0, 24.0, 23.0])

temperatures_4_villes = np.vstack([temperatures_villes, nouvelle_ville])
print(temperatures_4_villes)

[[18.5 20.  21.5 23.  22. ]
 [22.  24.5 26.  27.5 25. ]
 [25.  27.  29.5 31.  30. ]
 [19.  21.  22.  24.  23. ]]


<span style="color:blue">**QUESTION : Quelle est maintenant la forme de `temperatures_4_villes` ?**</span>

<span style="color:red">**CORRECTION :** La forme est `(4, 5)` : quatre villes et cinq jours.</span>

In [34]:
print(temperatures_4_villes.shape)

(4, 5)


## 8. Fonctions pré-existantes et vectorisation

Une idée essentielle de NumPy est d'éviter autant que possible les boucles Python lorsqu'une opération
peut être appliquée directement à un tableau entier.

Comparons une conversion de °C vers °F réalisée avec une boucle puis de manière vectorisée.

Formule : **°F = °C × 9/5 + 32**

In [35]:
def celsius_vers_fahrenheit_boucle(valeurs):
    sortie = np.empty(len(valeurs))
    for i in range(len(valeurs)):
        sortie[i] = valeurs[i] * 9 / 5 + 32
    return sortie

petit_tableau = np.array([10., 15., 20., 25., 30.])

print("Boucle :", celsius_vers_fahrenheit_boucle(petit_tableau))
print("NumPy  :", petit_tableau * 9 / 5 + 32)

Boucle : [50. 59. 68. 77. 86.]
NumPy  : [50. 59. 68. 77. 86.]


In [36]:
# Comparaison simple des temps d'exécution
from time import perf_counter

grand_tableau = rng.uniform(-10, 40, size=500_000)

t0 = perf_counter()
resultat_boucle = celsius_vers_fahrenheit_boucle(grand_tableau)
temps_boucle = perf_counter() - t0

t0 = perf_counter()
resultat_numpy = grand_tableau * 9 / 5 + 32
temps_numpy = perf_counter() - t0

print(f"Temps avec boucle : {temps_boucle:.4f} s")
print(f"Temps avec NumPy  : {temps_numpy:.4f} s")
print("Résultats identiques :", np.allclose(resultat_boucle, resultat_numpy))

Temps avec boucle : 0.0996 s
Temps avec NumPy  : 0.0010 s
Résultats identiques : True


<span style="color:blue">**QUESTION : Que fait le code précédent ? Qu'en déduisez-vous ?**</span>

<span style="color:red">**CORRECTION :** Les deux méthodes convertissent les températures de °C vers °F. La version vectorisée NumPy est généralement nettement plus rapide. Lorsqu'une opération doit être appliquée à tout un tableau, il faut donc chercher une opération NumPy plutôt qu'une boucle Python.</span>

Les opérations arithmétiques s'appliquent directement à chaque élément d'un tableau.

In [37]:
x = np.array([10, 15, 20, 25])

print("x      =", x)
print("x + 2  =", x + 2)
print("x - 5  =", x - 5)
print("x * 2  =", x * 2)
print("x / 2  =", x / 2)
print("x // 3 =", x // 3)

x      = [10 15 20 25]
x + 2  = [12 17 22 27]
x - 5  = [ 5 10 15 20]
x * 2  = [20 30 40 50]
x / 2  = [ 5.   7.5 10.  12.5]
x // 3 = [3 5 6 8]


In [38]:
anomalies = np.array([-3, -1, 1, 2])

print("Valeur absolue :", np.abs(anomalies))
print("Exponentielle  :", np.exp(anomalies))
print("Logarithme de la valeur absolue :", np.log(np.abs(anomalies)))

Valeur absolue : [3 1 1 2]
Exponentielle  : [0.04978707 0.36787944 2.71828183 7.3890561 ]
Logarithme de la valeur absolue : [1.09861229 0.         0.         0.69314718]


## 9. Masques booléens et `np.where`

Une comparaison appliquée à un tableau renvoie un tableau de booléens.
Ce masque peut ensuite servir à filtrer ou modifier les données.

In [39]:
mesures = np.array([
    [18.5, 22.0, 27.0],
    [24.5, 29.0, 31.5],
    [16.0, 25.5, 28.0]
])

print(mesures)
print("Température > 25 °C ?")
print(mesures > 25)

[[18.5 22.  27. ]
 [24.5 29.  31.5]
 [16.  25.5 28. ]]
Température > 25 °C ?
[[False False  True]
 [False  True  True]
 [False  True  True]]


<span style="color:blue">**QUESTION : Afficher uniquement les valeurs de `mesures` strictement supérieures à 25 °C.**</span>

<span style="color:red">**CORRECTION :** Le masque `mesures > 25` peut être utilisé directement dans les crochets : `mesures[mesures > 25]`.</span>

In [40]:
print(mesures[mesures > 25])

[27.  29.  31.5 25.5 28. ]


<span style="color:blue">**QUESTION : Avec `np.where`, conserver les températures supérieures ou égales à 25 °C et remplacer les autres par `-1`.**</span>

<span style="color:red">**CORRECTION :** `np.where(condition, valeur_si_vrai, valeur_si_faux)` permet ici d'écrire `np.where(mesures >= 25, mesures, -1)`.</span>

In [41]:
print(np.where(mesures >= 25, mesures, -1))

[[-1.  -1.  27. ]
 [-1.  29.  31.5]
 [-1.  25.5 28. ]]


## 10. Statistiques descriptives

NumPy propose de nombreuses fonctions statistiques : `sum`, `mean`, `min`, `max`, `std`,
`argmin`, `argmax`, `percentile`, etc.

Pour un tableau à plusieurs dimensions, l'argument `axis` permet de préciser la direction du calcul.

In [42]:
# Précipitations (mm) : lignes = villes, colonnes = jours
pluie = np.array([
    [0.0, 2.5, 0.0, 4.0, 1.0],
    [1.0, 0.0, 3.5, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 2.0]
])

print(pluie)
print("Pluie totale :", pluie.sum(), "mm")

[[0.  2.5 0.  4.  1. ]
 [1.  0.  3.5 0.  0. ]
 [0.  0.  0.  0.  2. ]]
Pluie totale : 14.0 mm


In [63]:
print("Somme par jour (axis=0) :", pluie.sum(axis=0))
print("Somme par ville (axis=1) :", pluie.sum(axis=1))

Somme par jour (axis=0) : [1.  2.5 3.5 4.  3. ]
Somme par ville (axis=1) : [7.5 4.5 2. ]


In [64]:
# La fonction sum existe aussi en Python, mais la version Numpy est beaucoup plus rapide
%timeit sum(pluie)
%timeit np.sum(pluie)

1.61 µs ± 9.49 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
1.37 µs ± 16.7 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


<span style="color:blue">**QUESTION : Calculer la température moyenne de chaque ville dans `temperatures_villes`.**</span>

<span style="color:red">**CORRECTION :** Chaque ville correspond à une ligne ; il faut donc moyenner les colonnes pour chaque ligne avec `axis=1`.</span>

In [44]:
print(temperatures_villes.mean(axis=1))

[21.  25.  28.5]


<span style="color:blue">**QUESTION : Calculer la température moyenne de chaque jour, toutes villes confondues.**</span>

<span style="color:red">**CORRECTION :** Chaque jour correspond à une colonne ; on réduit donc l'axe des lignes avec `axis=0`.</span>

In [45]:
print(temperatures_villes.mean(axis=0))

[21.83333333 23.83333333 25.66666667 27.16666667 25.66666667]


<span style="color:blue">**QUESTION : Trouver l'indice du jour le plus chaud pour chaque ville avec `np.argmax`.**</span>

<span style="color:red">**CORRECTION :** `np.argmax(temperatures_villes, axis=1)` renvoie, pour chaque ligne, l'indice de la colonne contenant le maximum.</span>

In [46]:
print(np.argmax(temperatures_villes, axis=1))

[3 3 3]


<span style="color:blue">**QUESTION : Calculer l'écart type des températures de chaque ville.**</span>

<span style="color:red">**CORRECTION :** `np.std(temperatures_villes, axis=1)` calcule la dispersion des températures de chaque ligne.</span>

In [47]:
print(np.std(temperatures_villes, axis=1))

[1.58113883 1.81659021 2.19089023]


## 11. Broadcasting

Le **broadcasting** désigne les règles qui permettent à NumPy d'effectuer des opérations
entre des tableaux de formes différentes lorsque leurs dimensions sont compatibles.

In [48]:
a = np.array([18, 20, 22])
b = np.array([2, 2, 2])

print("a + b =", a + b)

a + b = [20 22 24]


In [49]:
# Ajout d'une constante à tous les éléments
print("a + 5 =", a + 5)

a + 5 = [23 25 27]


Le broadcasting devient particulièrement utile avec les matrices.

Supposons qu'un capteur possède un biais différent pour chaque jour.
On veut corriger toutes les villes avec le même vecteur de corrections.

In [50]:
corrections_journalieres = np.array([-0.5, 0.0, 0.5, -1.0, 0.0])

print("Températures brutes :")
print(temperatures_villes)

print("\nTempératures corrigées :")
print(temperatures_villes + corrections_journalieres)

Températures brutes :
[[18.5 20.  21.5 23.  22. ]
 [22.  24.5 26.  27.5 25. ]
 [25.  27.  29.5 31.  30. ]]

Températures corrigées :
[[18.  20.  22.  22.  22. ]
 [21.5 24.5 26.5 26.5 25. ]
 [24.5 27.  30.  30.  30. ]]


<span style="color:blue">**QUESTION : Pourquoi peut-on additionner un tableau de forme `(3, 5)` et un tableau de forme `(5,)` ?**</span>

<span style="color:red">**CORRECTION :** NumPy considère le vecteur de 5 valeurs comme une ligne compatible avec les 5 colonnes de la matrice et le réplique implicitement pour les 3 lignes : c'est le broadcasting.</span>

<span style="color:blue">**QUESTION : Convertir en une seule instruction toutes les températures de `temperatures_villes` de °C vers °F.**</span>

<span style="color:red">**CORRECTION :** La formule s'applique élément par élément grâce à la vectorisation et au broadcasting des constantes : `temperatures_villes * 9/5 + 32`.</span>

In [51]:
temperatures_f = temperatures_villes * 9 / 5 + 32
print(temperatures_f)

[[65.3 68.  70.7 73.4 71.6]
 [71.6 76.1 78.8 81.5 77. ]
 [77.  80.6 85.1 87.8 86. ]]


## 12. Exercice de synthèse

On dispose maintenant de températures mesurées pendant 7 jours dans 4 villes.
Les lignes représentent les villes et les colonnes les jours.

L'objectif est de mobiliser plusieurs notions vues dans le TP.

In [52]:
villes = np.array(["Lille", "Paris", "Lyon", "Montpellier"])

temperature_semaine = np.array([
    [17.0, 18.5, 20.0, 19.0, 21.5, 22.0, 20.5],
    [19.0, 20.0, 22.5, 21.0, 24.0, 25.5, 23.0],
    [21.5, 23.0, 25.0, 24.5, 27.0, 29.0, 28.0],
    [24.0, 26.0, 28.5, 27.5, 30.0, 32.0, 31.0]
])

print(temperature_semaine)

[[17.  18.5 20.  19.  21.5 22.  20.5]
 [19.  20.  22.5 21.  24.  25.5 23. ]
 [21.5 23.  25.  24.5 27.  29.  28. ]
 [24.  26.  28.5 27.5 30.  32.  31. ]]


<span style="color:blue">**QUESTION : 1. Afficher la forme du tableau et son nombre total de mesures.**</span>

<span style="color:red">**CORRECTION :** La forme est `(4, 7)` et le tableau contient 28 mesures.</span>

In [53]:
print("Forme :", temperature_semaine.shape)
print("Nombre de mesures :", temperature_semaine.size)

Forme : (4, 7)
Nombre de mesures : 28


<span style="color:blue">**QUESTION : 2. Extraire les températures de Lyon.**</span>

<span style="color:red">**CORRECTION :** Lyon est sur la ligne d'indice 2.</span>

In [54]:
print(temperature_semaine[2, :])

[21.5 23.  25.  24.5 27.  29.  28. ]


<span style="color:blue">**QUESTION : 3. Extraire les températures du week-end (les deux derniers jours) pour toutes les villes.**</span>

<span style="color:red">**CORRECTION :** Les deux dernières colonnes s'obtiennent avec `[:, -2:]`.</span>

In [55]:
print(temperature_semaine[:, -2:])

[[22.  20.5]
 [25.5 23. ]
 [29.  28. ]
 [32.  31. ]]


<span style="color:blue">**QUESTION : 4. Calculer la moyenne hebdomadaire de chaque ville.**</span>

<span style="color:red">**CORRECTION :** Il faut calculer la moyenne sur les colonnes, donc utiliser `axis=1`.</span>

In [56]:
moyennes = temperature_semaine.mean(axis=1)
print(moyennes)

[19.78571429 22.14285714 25.42857143 28.42857143]


<span style="color:blue">**QUESTION : 5. Identifier la ville ayant la moyenne la plus élevée.**</span>

<span style="color:red">**CORRECTION :** On calcule les moyennes puis `np.argmax` donne l'indice de la plus grande ; cet indice permet d'accéder au nom dans `villes`.</span>

In [57]:
moyennes = temperature_semaine.mean(axis=1)
indice = np.argmax(moyennes)
print(villes[indice], moyennes[indice])

Montpellier 28.428571428571427


<span style="color:blue">**QUESTION : 6. Compter le nombre total de mesures strictement supérieures à 25 °C.**</span>

<span style="color:red">**CORRECTION :** Le masque produit des booléens ; `np.sum` compte les `True` car ils valent 1.</span>

In [58]:
print(np.sum(temperature_semaine > 25))

10


<span style="color:blue">**QUESTION : 7. Créer un nouveau tableau où toutes les températures supérieures à 30 °C sont plafonnées à 30 °C avec `np.where`.**</span>

<span style="color:red">**CORRECTION :** `np.where(temperature_semaine > 30, 30, temperature_semaine)` remplace uniquement les valeurs dépassant 30.</span>

In [59]:
temperature_plafonnee = np.where(temperature_semaine > 30, 30, temperature_semaine)
print(temperature_plafonnee)

[[17.  18.5 20.  19.  21.5 22.  20.5]
 [19.  20.  22.5 21.  24.  25.5 23. ]
 [21.5 23.  25.  24.5 27.  29.  28. ]
 [24.  26.  28.5 27.5 30.  30.  30. ]]


<span style="color:blue">**QUESTION : 8. Une calibration impose les corrections `[0.0, -0.5, 0.0, 0.5, 0.0, -1.0, 0.0]` selon le jour. Appliquer ces corrections à toutes les villes grâce au broadcasting.**</span>

<span style="color:red">**CORRECTION :** Le vecteur contient 7 valeurs, comme le nombre de colonnes. Il est donc diffusé sur chacune des 4 lignes.</span>

In [60]:
corrections = np.array([0.0, -0.5, 0.0, 0.5, 0.0, -1.0, 0.0])
temperature_corrigee = temperature_semaine + corrections
print(temperature_corrigee)

[[17.  18.  20.  19.5 21.5 21.  20.5]
 [19.  19.5 22.5 21.5 24.  24.5 23. ]
 [21.5 22.5 25.  25.  27.  28.  28. ]
 [24.  25.5 28.5 28.  30.  31.  31. ]]


## Pour aller plus loin

Quelques pistes à tester :

- calculer le 75ᵉ percentile des températures avec `np.percentile` ;
- rechercher la température minimale de chaque jour ;
- compter le nombre de jours à plus de 25 °C pour chaque ville ;
- normaliser les températures en retirant à chaque ville sa propre moyenne ;
- ajouter une cinquième ville avec `np.vstack`.

L'objectif n'est pas de mémoriser toutes les fonctions NumPy, mais de prendre l'habitude de raisonner
en **tableaux, formes, axes, masques et opérations vectorisées**.